# INTERPOLATIONS 




## DIVIDED DIFFERENCE 

In [ ]:

def divided_differences(nodes, values):
    ### input: list of nodes [x0, x1, ..., xn] length n+1 & list of values [f(x0), f(x1), ..., f(xn)] length n+1
    ### Output: list of divided differences [f[x0], f[x0, x1], ..., f[x0, x1, ..., xn]] length n+1
    dd = values.copy()

    result = [dd[0]]

    for k in range(1, len(nodes)):
        for i in range(len(nodes) - k):
            dd[i] = (dd[i + 1] - dd[i]) / (nodes[i + k] - nodes[i])

        result.append(dd[0])

    return result

    

## LAGRANGE INTERPOLATION 

In [ ]:
def lagrange_interpolation(x_values: list, y_values: list, x: int):
    """
    Computes the Lagrange interpolation polynomial at the point x.

    x_values: list of interpolation nodes, e.g. [0, 1, 2]
    y_values: list of function values, e.g. [1, 3, 2]
    x: the point where we want to evaluate the polynomial
    """
    n = len(x_values)
    result = 0

    for i in range(n):
        # Start computing L_i(x)
        L_i = 1

        for j in range(n):
            if j != i:
                L_i = L_i * (x - x_values[j]) / (x_values[i] - x_values[j])

        # Add y_i * L_i(x) to the result
        result = result + y_values[i] * L_i

    return result

## NEWTON INTERPOLATION 

In [ ]:
def newton_coefficients(x_values, y_values):
    """
    Computes the Newton divided difference coefficients.

    x_values: interpolation nodes
    y_values: function values

    returns: list of Newton coefficients
    """

    n = len(x_values)

    # Start with a copy of y_values
    coefficients = y_values.copy()

    # Compute divided differences
    for k in range(1, n):
        for i in range(n - 1, k - 1, -1):
            numerator = coefficients[i] - coefficients[i - 1]
            denominator = x_values[i] - x_values[i - k]
            coefficients[i] = numerator / denominator

    return coefficients

def newton_interpolation(x_values, y_values, x):
    """
    Evaluates the Newton interpolation polynomial at x.
    """

    coefficients = newton_coefficients(x_values, y_values)

    n = len(x_values)

    # Start with the highest coefficient
    result = coefficients[n - 1]

    # Nested multiplication
    for i in range(n - 2, -1, -1):
        result = result * (x - x_values[i]) + coefficients[i]

    return result

## NEVILLE INTERPOLATION (extention of newton)

In [ ]:
def neville_interpolation(x_values, y_values, x):
    """
    Evaluates the interpolation polynomial at x using Neville's algorithm.
    """

    n = len(x_values)

    # Start with the y-values
    p = y_values.copy()

    # Build Neville table in-place
    for k in range(1, n):
        for i in range(n - k):
            left = (x_values[i + k] - x) * p[i]
            right = (x - x_values[i]) * p[i + 1]
            denominator = x_values[i + k] - x_values[i]

            p[i] = (left + right) / denominator

    return p[0]

## HERMITE INTERPOLATION 


In [ ]:
import numpy as np 
import matplotlib.pyplot as plt

def f(x): #our function 
    return np.sin(x)

def df(x): #our derivative 
    return np.cos(x)

def lagrange_basis(k, x_grid, x):
    """
    Computes the k-th Lagrange basis polynomial l_k(x).
    """
    result = 1.0

    for j in range(len(x_grid)):
        if j != k:
            result *= (x - x_grid[j]) / (x_grid[k] - x_grid[j])

    return result

def lagrange_basis_derivative_at_node(k, x_grid):
    """
    Computes l_i'(x_i).
    Formula: l_i'(x_k) = sum_{j != i} 1 / (x_i - x_j)
    """
    result = 0.0

    for j in range(len(x_grid)):
        if j != k:
            result += 1 / (x_grid[k] - x_grid[j])

    return result


def HermiteInterpolationFormula(x_grid, y_grid, dy_grid, x_eval):
    """
    Inputs:
        x_grid  = interpolation nodes x_k
        y_grid  = function values f(x_k)
        dy_grid = derivative values f'(x_k)
        x_eval  = points where we evaluate the Hermite polynomial
    """

    hermite_values = []

    for x in x_eval:
        H = 0.0

        for k in range(len(x_grid)):
            x_k = x_grid[k]

            # Lagrange basis l_k(x)
            l_k = lagrange_basis(k, x_grid, x)

            # Derivative l_k'(x_k)
            dl_k = lagrange_basis_derivative_at_node(k, x_grid)

            # Hermite basis functions
            A = (1 - 2 * (x - x_k) * dl_k) * l_k**2
            B = (x - x_k) * l_k**2

            # Add contribution from f(x_k) and f'(x_k)
            H += y_grid[k] * A + dy_grid[k] * B

        hermite_values.append(H)

    return np.array(hermite_values)


#PLOTTING 
a = 0
b = np.pi

n = 4

x_grid = np.linspace(a, b, n + 1)
y_grid = f(x_grid)
dy_grid = df(x_grid)

x_eval = np.linspace(a, b, 1000)
y_true_values = f(x_eval)

hermite_values = HermiteInterpolationFormula(x_grid, y_grid, dy_grid, x_eval)

plt.figure(figsize=(8, 6))
plt.plot(x_eval, y_true_values, label="Original function")
plt.plot(x_eval, hermite_values, label="Hermite interpolation")
plt.xlabel("x")
plt.ylabel("y")
plt.title(r"Hermite interpolation of $f(x)=\sin(x)$")
plt.legend()
plt.show()

## ERROR CONVERGENCE RATE/SPEED (Lagrange, Newton, Nevilles)

In [ ]:
import numpy as np 
import matplotlib.pyplot as plt
a = 0
b = 1

n_list = [2, 4, 8, 12, 16, 20]
x_eval = np.linspace(a, b, 1000)
y_true_values = f(x_eval) #the function with x 

errors_list = []

plt.figure(figsize=(8, 6))

for n in n_list:
    x_grid = np.linspace(a, b, n + 1)
    y_grid = f(x_grid)

    lagrange_values = LagrangeInterpolation(x_grid, y_grid, x_eval) #SAME FOR NEWTON AND NEVILLE 

    error = np.max(np.abs(y_true_values - lagrange_values))
    errors_list.append(error)

    plt.plot(x_eval, lagrange_values, label=f"n={n}")

plt.xlabel("x")
plt.ylabel("y")
plt.title(r"Lagrange interpolation of $f(x)=x^{1/4}$ on $[0,1]$")
plt.legend()
plt.show()


errors_array = np.array(errors_list)
n_array = np.array(n_list)

plt.figure(figsize=(8, 6))
plt.loglog(n_array, errors_array, "o-", label="Interpolation error")
plt.xlabel("n")
plt.ylabel("Maximum error")
plt.title("Log-log plot of Lagrange interpolation error")
plt.legend()
plt.show()

### ALGEBRAIC CONVERGENCE (loglog plot)

In [ ]:
errors_array = np.array(errors_list)
n_array = np.array(n_list)
# We assume error ~ C * n^(-p)

# FORMULA OF THE ERROR ESTIMATE = log(e_old / e_new) / log(n_new / n_old)
p_values = []

for k in range(len(n_array) - 1):
    e_old = errors_array[k]
    e_new = errors_array[k + 1]

    n_old = n_array[k]
    n_new = n_array[k + 1]

    p_k = np.log(e_old / e_new) / np.log(n_new / n_old)
    p_values.append(p_k)

p_values = np.array(p_values)

# Use the last observed convergence rate as estimate
p_estimated = p_values[-1]

reference = errors_array[-1] * (n_array / n_array[-1])**(-p_estimated)

print("Observed convergence rates:")
for k in range(len(p_values)):
    print(f"from n={n_array[k]} to n={n_array[k+1]}: p ≈ {p_values[k]:.4f}")

print(f"Estimated convergence speed: algebraic with p = {p_estimated:.4f}")

plt.figure(figsize=(8, 6))
plt.loglog(n_array, errors_array, "o-", label="Interpolation error")
plt.loglog(n_array, reference, "--", label=fr"Reference slope $n^{{-{p_estimated:.2f}}}$")
plt.xlabel("n")
plt.ylabel("Maximum error")
plt.title("Log-log plot of interpolation error")
plt.legend()
plt.show()

### EXPONENTIAL ERROR CONVERGENCE  (semilog plot)

In [ ]:
errors_array = np.array(errors_list)
n_array = np.array(n_list)

# Manual exponential convergence estimate
# FORMULA USED =  C * exp(-alpha * n)

alpha_values = []

for k in range(len(n_array) - 1):
    e_old = errors_array[k]
    e_new = errors_array[k + 1]

    n_old = n_array[k]
    n_new = n_array[k + 1]

    alpha_k = np.log(e_old / e_new) / (n_new - n_old)
    alpha_values.append(alpha_k)

alpha_values = np.array(alpha_values)

# Use the last observed rate as estimate
alpha_estimated = alpha_values[-1]

# Equivalent rho value: error ~ C * rho^(-n)
rho_estimated = np.exp(alpha_estimated)

reference = errors_array[-1] * np.exp(-alpha_estimated * (n_array - n_array[-1]))

print("Observed exponential convergence rates:")
for k in range(len(alpha_values)):
    print(
        f"from n={n_array[k]} to n={n_array[k+1]}: "
        f"alpha ≈ {alpha_values[k]:.4f}, rho ≈ {np.exp(alpha_values[k]):.4f}"
    )

print(f"Estimated exponential convergence speed: alpha = {alpha_estimated:.4f}")
print(f"Equivalent form: error ~ C * rho^(-n), with rho = {rho_estimated:.4f}")
plt.figure(figsize=(8, 6))
plt.semilogy(n_array, errors_array, "o-", label="Interpolation error")
plt.semilogy(n_array, reference, "--", label=fr"Reference $e^{{-{alpha_estimated:.2f}n}}$")
plt.xlabel("n")
plt.ylabel("Maximum error")
plt.title("Semilog plot of interpolation error")
plt.legend()
plt.show()

## SPLINES INTERPOLATION  

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
a = 0 # OUR INTERVAL 
b = 1 

x_eval = np.linspace(a, b, 10001)
y_true_values = x_eval**(1/4) # OUR FUNCTION 

def LinearSplines(set_ordered_gridpoints, values_at_gridpoints, set_evaluationpoints):
    x_grid = set_ordered_gridpoints
    y_grid = values_at_gridpoints
    x_eval = set_evaluationpoints
    splinevalues = []

#finding the interval where x lies 
    for x in x_eval:
        if x <= x_grid[0]:
            i = 0
        elif x >= x_grid[-1]:
            i = len(x_grid) - 2
        else:
            i = 0
            for j in range(len(x_grid) - 1):
                if x_grid[j] <= x <= x_grid[j + 1]:
                    i = j
                    break
#setting left and right x, y from the point for the formula 
        x_left, x_right = x_grid[i], x_grid[i+1]
        y_left, y_right = y_grid[i], y_grid[i+1]
#hat basis function 
        b_left = (x_right - x) / (x_right - x_left)
        b_right = (x - x_left) / (x_right - x_left)

        y_interpolated = y_left * b_left + y_right * b_right # Σf(x_i) * b_i(x) the formula 
        splinevalues.append(y_interpolated)

    return np.array(splinevalues)

plt.figure(figsize=(12, 8))
plt.plot(x_eval, y_true_values, label=r'$f(x)=x^{1/4}$', linewidth=2)

### ERROR CONVERGENCE RATE LINEAR SPLINES 

In [ ]:
errors_list = []
power = range(2, 11)
n_list = [2**k for k in power]
for n in n_list:
    x_grid = np.linspace(a, b, n + 1)
    y_grid = x_grid**(1/4)

    linear_values = LinearSplines(x_grid, y_grid, x_eval)  #our interpolation 
    error = np.max(np.abs(y_true_values - linear_values)) #error formula 

    errors_list.append(error) 

    plt.plot(x_eval, linear_values, label=f'n={n}')

plt.xlabel("x")
plt.ylabel("y")
plt.title("Linear spline interpolation of $f(x)=x^{1/4}$ on [0,1]")
plt.legend()
plt.show()


### ERROR CONVERGENCE SPEED for linear splines 

To determine the speed of convergence, you need to plot the reference convergence
slopes Eref,m(n) =(1/n)^ m = h^m,

In [ ]:
errors_array = np.array(errors_list)
reference = errors_array[-1] * (n_list[-1] / np.array(n_list))**2 #TAKING THE LAST POINT  

plt.figure(figsize=(8, 6))
plt.loglog(n_list, errors_array, label='Interpolation error')
plt.loglog(n_list, reference, label=r'Reference slope $n^{-1/4}$')
plt.xlabel('n')
plt.ylabel('Maximum error')
plt.title('Log-log plot of interpolation error')
plt.legend()
plt.show()

print("Convegrence speed of the errors is algebraic with m= 2")

## PIECEWIESE CONSTANT SPLINE APPROXIMATION 

In [ ]:
def piecewise_constant_approximation(set_ordered_gridpoints, set_evaluationpoints ):
    x_grid = set_ordered_gridpoints #nodes
    x_eval = set_evaluationpoints #evaluation points 
    approximation=[]
    
     # x_grid looks like [x0, x1, ... x{n-1}]
    for x in x_eval: #go through every values of x we want to interpolate
        if x <= x_grid[0]:  # if x is smaller or equal to the 1st grid point
            i = 0           #interval index is 0, we use the 1st interval [x_grid[0], x_grid[1]]
        elif x >= x_grid[-1]:  # if x is greater or equal to the last grid point
            i = len(x_grid) - 2 #since indeces are from 0 to n-1 the last interval is [x_{n-2}, x_{n-1}] 
        else: #now points inside of the domain 
            i = 0 #will be overwritten 
            for j in range(len(x_grid) - 1): 
                if x_grid[j] <= x <= x_grid[j + 1]: #check if x is inside
                    i = j #gives the index for the interval 
                    break
        x_left, x_right= x_grid[i], x_grid[i+1] #interval where x lies 
        y_interpolated= (4/3)*((x_right**(3/4) - x_left**(3/4))/(x_right-x_left)) #OVERALL CELL AVERAGE (comes from integral over the interval)
        approximation.append(y_interpolated)
        
    return approximation  

#PLOT 
f_opt= piecewise_constant_approximation(x_grid, x_eval)


plt.figure(figsize=(12, 8))
plt.plot(x_eval, y_true_values, label=r"$f(x)=x^{-1/4}$")
plt.step(x_eval, f_opt, label=r"$f_{\mathrm{opt}}$")

plt.xlabel("x grid ")
plt.ylabel("y grid")
plt.title(" f_optimal")
plt.legend()
plt.show()


# DIFFERENTIATION 


##  GENERAL CASE 

In [ ]:
def divided_differences_coefficients(x_nodes, y_nodes):
    """
    Computes Newton divided difference coefficients.
    """


    x_nodes = np.array(x_nodes, dtype=float)
    coeffs = np.array(y_nodes, dtype=float)

    n = len(x_nodes)

    for level in range(1, n):
        for i in range(n - 1, level - 1, -1):
            coeffs[i] = (coeffs[i] - coeffs[i - 1]) / (x_nodes[i] - x_nodes[i - level])

    return coeffs

def general_derivative_approximation(f, x_nodes):
    """
    Approximates f'(x0), where x0 = x_nodes[0]
    Uses the general formula from Theorem 2.17.
    """

    x_nodes = np.array(x_nodes, dtype=float)
    y_nodes = f(x_nodes)

    coeffs = divided_differences_coefficients(x_nodes, y_nodes)

    x0 = x_nodes[0]

    derivative_approx = 0.0

    product = 1.0

    for k in range(1, len(x_nodes)):
        derivative_approx += product * coeffs[k]

        # Update product:
        # after k=1, product becomes (x0 - x1)
        # after k=2, product becomes (x0 - x1)(x0 - x2)
        product *= (x0 - x_nodes[k])

    return derivative_approx

## FORWARD DIFFERENCE 

In [ ]:
def forward_difference(f, x0, h):

    return (f(x0 + h) - f(x0)) / h

## SYMMETRIC/CENTRAL DIFFERENCE 
used when x0 is an interior point: x-1 and x1 exist 

In [ ]:
def central_difference(f, x0, h):

    return (f(x0 + h) - f(x0 - h)) / (2 * h)

## ONE SIDED FORWARD DIFFERENCE 
used with 3 points when x0 is at the left boundary 

In [ ]:
def forward_difference_3point(f, x0, h):
    
    return (-3 * f(x0) + 4 * f(x0 + h) - f(x0 + 2*h)) / (2 * h)

## ERRORS CONVERGENCE 

In [ ]:
import matplotlib.pyplot as plt

def derivative_error_convergence(f, df, x0, h_list):
    errors_forward = []
    errors_central = []
    errors_forward_3point = []

    true_value = df(x0)

    for h in h_list:
        approx_forward = forward_difference(f, x0, h)
        approx_central = central_difference(f, x0, h)
        approx_forward_3point = forward_difference_3point(f, x0, h)

        errors_forward.append(abs(true_value - approx_forward))
        errors_central.append(abs(true_value - approx_central))
        errors_forward_3point.append(abs(true_value - approx_forward_3point))

    return np.array(errors_forward), np.array(errors_central), np.array(errors_forward_3point)

plt.figure(figsize=(8, 6))
plt.loglog(h_list, errors_forward, "o-", label="Forward difference")
plt.loglog(h_list, errors_central, "o-", label="Central difference")
plt.loglog(h_list, errors_forward_3point, "o-", label="3-point forward difference")
plt.xlabel("h")
plt.ylabel("Absolute error")
plt.title("Error convergence of finite difference formulas")
plt.legend()
plt.show()

# INTEGRATION 

## MIDPOINT + error 

In [ ]:
def any_function(f, x):
    return f(x)
f=lambda x: x**(1/4)


def composite_midpoint(a, b, f, n):
    sum=0
    eval_points=np.linspace(a, b, n+1)
    for i in range(len(eval_points)-1):
        mid_points= (eval_points[i+1]+eval_points[i])/2
        value= any_function(f, mid_points)
        sum+=value*((b-a)/n)
    return  sum

def error_midpoint(a, b, f, n, exact_value):
    approx = composite_midpoint(a, b, f, n)
    return abs(approx - exact_value)
for n in [2**1, 2**2, 2**3, 2**4, 2**5, 2**6, 2**7, 2**8, 2**9, 2**10]:
    print(n, error_midpoint(0, 1, f, n, 0.8))

## TRAPEZOID + error 

In [ ]:
def any_function(f, x):
    return f(x)
f=lambda x: x**(1/4)

def composite_trapezoid(a, b, f, n):
    sum=0
    eval_points=np.linspace(a, b, n+1)
    for i in range(len(eval_points)-1):
        mid_values= (any_function(f, eval_points[i+1])+any_function(f, eval_points[i]))/2
        sum+=mid_values*((b-a)/n)
    return  sum


def error_trapezoid(a, b, f, n, exact_value):
    approx = composite_trapezoid(a, b, f, n)
    return abs(approx - exact_value)
for n in [2**1, 2**2, 2**3, 2**4, 2**5, 2**6, 2**7, 2**8, 2**9, 2**10]:
    print(n, error_trapezoid(0, 1, f, n, 0.8))

## SIMPSONS FORMULA + error 

In [ ]:
def any_function(f, x):
    return f(x)
f=lambda x: x**(1/4)

def composite_simpsons(a, b, f, n):
    sum=0
    eval_points=np.linspace(a, b, n+1)
    for i in range(n//2):
        sumvalues= (any_function(f, eval_points[2*i])+4*any_function(f, eval_points[2*i+1]) +any_function(f, eval_points[2*i+2]))/6
        sum+=sumvalues*2*((b-a)/n)
    return  sum


def error_simpsons(a, b, f, n, exact_value):
    approx = composite_simpsons(a, b, f, n)
    return abs(approx - exact_value)
for n in [2**1, 2**2, 2**3, 2**4, 2**5, 2**6, 2**7, 2**8, 2**9, 2**10]:
    print(n, error_simpsons(0, 1, f, n, 0.8))

## LEFT + error 

In [ ]:

def composite_left_rectangle(a, b, f, n):

    x_points = np.linspace(a, b, n + 1)  # x_0, x_1, ..., x_n
    total = 0
    
    for j in range(n):  # j = 0 to n-1
        total += any_function(f, x_points[j]) * (x_points[j+1] - x_points[j])
    
    return total

def error_left(a, b, f, n, exact_value):
    
    approx = composite_left_rectangle(a, b, f, n)
    return abs(approx - exact_value)    
for n in [2**1, 2**2, 2**3, 2**4, 2**5, 2**6, 2**7, 2**8, 2**9, 2**10]:
    print(n, error_left(0, 1, f, n, 0.8))

## RIGHT + error 

In [ ]:
def composite_right_rectangle(a, b, f, n):
    x_points = np.linspace(a, b, n + 1)  # x_0, x_1, ..., x_n
    total = 0
    
    for j in range(n):  # j = 0 to n-1
        total += any_function(f, x_points[j+1]) * (x_points[j+1] - x_points[j])
    
    return total

def error_right(a, b, f, n, exact_value):
    
    approx = composite_right_rectangle(a, b, f, n)
    return abs(approx - exact_value)    
for n in [2**1, 2**2, 2**3, 2**4, 2**5, 2**6, 2**7, 2**8, 2**9, 2**10]:
    print(n, error_right(0, 1, f, n, 0.8))

## QUADRATURE 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# Function from the exercise
def f(x):
    return np.exp(np.cos(x)) * np.log(x)


# Exact value given in the exercise
I_exact = -2.5832789043697337


# Two-point Gauss nodes on [0, 1] 
xi1 = 1/2 - np.sqrt(3)/6
xi2 = 1/2 + np.sqrt(3)/6

# Gauss weights on [0, 1], else w=1 on [-1, 1] 
w1 = 1/2
w2 = 1/2


def composite_gauss_2(n):
    total = 0
    
    h = 1 / n
    
    for i in range(n):
        left = i * h
        
        # Map the Gauss nodes from [0,1] to the small interval [left, left+h]
        x1 = left + h * xi1
        x2 = left + h * xi2
        
        # Add the contribution of this subinterval
        total += h * (w1 * f(x1) + w2 * f(x2))
    
    return total

# NON LINEAR EQUATIONS 

## BISECTION METHOD 

In [ ]:
def SignChange(f):
    # check  [-1,1] with nodes {-1,0,1}
    if f(-1)*f(0) < 0:
        return (-1, 0)
    elif f(0)*f(1) < 0:
        return (0, 1)

    n = 2
    while True:
        for i in range(0, n**2):
            x_left = -n + (2 * i) / n
            x_right = -n + (2 * (i + 1)) / n

            if f(x_left)*f(x_right)< 0:
                return (x_left, x_right)

        n = n + 1

print(SignChange(lambda x: x-2))

# eps  tolerance level or 
# also n - number of steps that controlls how good the method is 
def bisection(f, a, b, eps):
    n = 0
    
    while (b - a) / (2**n) > eps:
        m = (a + b) / 2
        
        if f(m) == 0:
            return m
        
        if f(a) * f(m) < 0:
            b = m
        else:
            a = m
        
        n = n + 1
    
    return (a + b) / 2

root = bisection(lambda x:  ((x-10)**7)*((x-4)**4)-5, 10, 11, 10**-12)
print(root)


## NEWTON METHOD + error 

In [ ]:
def newton_method(f, df, x0, tolerance=1e-10, max_iterations=100):
    x = x0
    
    approximations = [x]
    
    for k in range(max_iterations):
        fx = f(x)
        dfx = df(x)
        
        # Avoid division by zero
        if dfx == 0:
            print("Derivative is zero. Newton method stops.")
            return x, approximations
        
        # Newton formula
        x_new = x - fx / dfx
        
        approximations.append(x_new)
        
        # Stopping condition
        if abs(x_new - x) < tolerance:
            print("Newton method converged.")
            print("Number of iterations:", k + 1)
            return x_new, approximations
        
        x = x_new
    
    print("Newton method did not converge within the maximum number of iterations.")
    return approximations

#Example usage 

def f(x):
    return x**2 - 2

def df(x):
    return 2*x

# Error plotting 

root, approximations = newton_method(f, df, x0=1)
errors = [] 

print("Approximate root:", root)
print("f(root):", f(root))
exact_root = np.sqrt(2)

for x in approximations:
    error = abs(x - exact_root)
    errors.append(error)

plt.figure(figsize=(8, 5))
plt.semilogy(errors, marker="o")
plt.xlabel("iteration k")
plt.ylabel("error")
plt.title("Newton method convergence")
plt.grid(True)
plt.show()

## NEWTON'S METHOD FOR 2 EQUATIONS 

In [ ]:
import math 

# Define F(x, y)
# This returns the vector F = [f1, f2]

def F(x, y):
    f1 = x**2 + y**2 - 1
    f2 = x - y
    return [f1, f2]


# Define the Jacobian matrix J(x, y)
# J = [[df1/dx, df1/dy],
#      [df2/dx, df2/dy]]

def J(x, y):
    return [
        [2*x, 2*y],
        [1, -1]
    ] 

def inverse_2x2(A):
    a = A[0][0]
    b = A[0][1]
    c = A[1][0]
    d = A[1][1]
    
    determinant = a*d - b*c
    
    if determinant == 0:
        print("Matrix is not invertible.")
        return None
    
    A_inverse = [
        [d / determinant, -b / determinant],
        [-c / determinant, a / determinant]
    ]
    
    return A_inverse

def matrix_vector_multiply(A, v):
    result_1 = A[0][0] * v[0] + A[0][1] * v[1]
    result_2 = A[1][0] * v[0] + A[1][1] * v[1]
    
    return [result_1, result_2]


def newton_system_2x2(x0, y0, tolerance=1e-10, max_iterations=20):
    x = x0
    y = y0
    
    for k in range(max_iterations):
        # Compute F(x, y)
        F_value = F(x, y)
        
        # Compute J(x, y)
        J_value = J(x, y)
        
        # Compute inverse of J
        J_inverse = inverse_2x2(J_value)
        
        if J_inverse is None:
            print("Newton method stopped.")
            return x, y
        
        # Compute J^{-1} F
        correction = matrix_vector_multiply(J_inverse, F_value)
        
        # Newton update
        x_new = x - correction[0]
        y_new = y - correction[1]
        
        print("Iteration:", k + 1)
        print("x =", x_new)
        print("y =", y_new)
        print("F(x,y) =", F(x_new, y_new))
        print()
        
        # Stop if the change is very small
        change = math.sqrt((x_new - x)**2 + (y_new - y)**2)
        
        if change < tolerance:
            print("Newton method converged.")
            return x_new, y_new
        
        x = x_new
        y = y_new
    
    print("Maximum number of iterations reached.")
    return x, y

## FIXED POINT ITERATION 

In [ ]:
def fixed_point_iteration(phi, x0, tol=1e-10, max_iter=100):
    x = x0
    
    for k in range(max_iter):
        x_new = phi(x)
        
        if abs(x_new - x) < tol:
            return x_new
        
        x = x_new
    
    return x

# LINEAR SYSTEMS 

## DIRECT METHODS 


### LOWER TRIANGULAR MATRICES 

In [4]:
def solve_lower_triangular(L, b):
    n = len(b)
    x = np.zeros(n)
    
    for i in range(n):
        s = b[i]
        
        for j in range(i):
            s = s - L[i][j] * x[j]
        
        x[i] = s / L[i][i]
    
    return x

L = np.array([
    [2, 0, 0],
    [3, 1, 0],
    [1, -2, 4]
], dtype=float)

b = np.array([4, 7, 3], dtype=float)

x = solve_lower_triangular(L, b)

print(x)


[2.   1.   0.75]


### UPPER TRIANGULAR MATRICES 

In [3]:
def solve_upper_triangular(U, b):
    n = len(b)
    x = np.zeros(n)
    
    for i in range(n - 1, -1, -1):
        s = b[i]
        
        for j in range(i + 1, n):
            s = s - U[i][j] * x[j]
        
        x[i] = s / U[i][i]
    
    return x

U = np.array([
    [2, 3, 1],
    [0, 1, -2],
    [0, 0, 4]
], dtype=float)

b = np.array([10, -3, 8], dtype=float)

x = solve_upper_triangular(U, b)

print(x)


[2.5 1.  2. ]


### QR DECOMPOSITION + GRAMM SCHMIDT 

In [ ]:
import numpy as np 
def QR_decomp(A):
    n=len(A)
    Q=[[0 for j in range(n)] for i in range(n)] #Q matrix only with 0 entries 
    for i in range(n): #Q matrix = Id
        Q[i][i] = 1
    
    Q = np.array(Q, dtype=float) #convert into a numpy array because matrix multiplication 
    R = np.array(A, dtype=float)

    sdot = 0

    for k in range(n-1):
        r_k = R[k:n, k] # from row k to row n-1 take column kth
        norm_r = np.linalg.norm(r_k)
        sdot = 1 / (norm_r * (r_k[0] + norm_r))
        
        #Constructing a Householder matrix 
        H_k = np.eye(n - k) - sdot * np.outer(r_k, r_k) # ID- multiplication H*H^T

        # update R[k:n, k:n] = H_k R[k:n, k:n]
        R[k:n, k:n] = np.matmul(H_k, R[k:n, k:n]) # matrix multiplication 

        # update Q[:, k:n] = Q[:, k:n] H_k
        Q[:, k:n] = np.matmul(Q[:, k:n], H_k)

    return Q, R

### QR DECOMPOSITION + HOUSEHOLDER TRANSFORMATION 

In [ ]:
import numpy as np


def householder_qr(A):
    A = A.astype(float)
    
    m, n = A.shape
    
    Q = np.eye(m)
    R = A.copy()
    
    for k in range(min(m, n)):
        # Take the part of column k below and including the diagonal
        x = R[k:, k]
        
        norm_x = np.linalg.norm(x)
        
        if norm_x == 0:
            continue
        
        # Choose sign for numerical stability
        if x[0] >= 0:
            sign = 1
        else:
            sign = -1
        
        # e1 = (1, 0, 0, ...)
        e1 = np.zeros(len(x))
        e1[0] = 1
    
        # Householder vector
        v = x + sign * norm_x * e1
        
        # Normalize v
        v = v / np.linalg.norm(v)
        
        # Apply Householder reflection to R
        R[k:, k:] = R[k:, k:] - 2 * np.outer(v, v @ R[k:, k:])
        
        # Accumulate Q
        Q[:, k:] = Q[:, k:] - 2 * np.outer(Q[:, k:] @ v, v)
    
    return Q, R

Identity matrix 

In [ ]:
import numpy as np 
np.eye(n)  


Matrix * Transpose matrix

In [ ]:
A= [ [1, 0],
    [ 0, 1]]
np.outer(A, A)